# One design on Colab

Install the public package, generate one enzyme design, and download the result.
No repository checkout is needed. This notebook runs generation only; refolding and
experimental validation are separate. See the full [enzyme design tutorial](enzyme_design_tutorial.ipynb).


1. Select **Runtime → Change runtime type → GPU** (A100 recommended) and a
   **Python 3.12** runtime. If necessary, choose **Runtime Version → 2026.07**;
   see [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html).
2. Run the installation cell. It uses the public `rfd4-proteina==0.0.3` package
   and matching public tutorial inputs; no GitHub key or Hugging Face token is needed.
3. When it says **Installation complete**, select **Runtime → Restart session**,
   then rerun that cell. Continue only after it prints **Ready**. Repeating setup
   in the same runtime reuses the installation.

The model downloads from Hugging Face on first generation. T4 is not qualified hardware;
GPU availability and memory vary. Save outputs before deleting a runtime.


<div class="cloud-launch-box">
<div class="cloud-launch-title">🚀 Run this on a Colab GPU</div>
<div class="badge-group">
<a href="https://colab.research.google.com/github/RosettaCommons/RFD4-Proteina/blob/main/docs/examples/colab_quickstart.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>
</div>
</div>


In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import json
    import os
    import re
    import shutil
    import subprocess
    import urllib.request
    from pathlib import Path

    if sys.version_info[:2] != (3, 12):
        raise SystemExit(
            f"Python 3.12 is required; this runtime has {sys.version.split()[0]}. "
            "Select Runtime > Change runtime type > Runtime Version > 2026.07, then reconnect. "
            "See https://research.google.com/colaboratory/runtime-version-faq.html"
        )
    if not shutil.which("nvidia-smi"):
        raise SystemExit("Select a GPU in Runtime > Change runtime type, then reconnect.")
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
    cuda = re.search(r"CUDA Version:\s*(\d+)", smi.stdout)
    if not cuda or int(cuda.group(1)) < 12:
        raise SystemExit("RFD4-Proteina requires a driver supporting CUDA 12 or newer.")
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip().splitlines()[0]
    capability = float(gpu.rsplit(",", 1)[1])
    major = 13 if int(cuda.group(1)) >= 13 else 12
    backend = "cu130" if major == 13 else "cu126"
    requirements = [
        f"rfd4-proteina[all-cuda{major}]==0.0.3", f"torch==2.14.1+{backend}",
        "torchvision", "torchaudio", "py3Dmol",
    ]
    print(f"GPU: {gpu}; installing CUDA {major} dependencies ({backend})")
    if capability < 8:
        print("T4 and other pre-Ampere GPUs are unqualified; use A100 for the supported hardware path.")

    state_file = Path("/content/.rfd4-colab-setup.json")
    state = json.loads(state_file.read_text()) if state_file.exists() else {}
    if state and state["requirements"] != requirements:
        raise SystemExit("The CUDA profile changed. Select Runtime > Disconnect and delete runtime, then rerun setup.")
    if not state:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.13.4"], check=True)
        subprocess.run([
            sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable,
            "--torch-backend", backend, "--only-binary", "tmol", *requirements,
        ], check=True)
        state = {"requirements": requirements, "pid": os.getpid()}
        state_file.write_text(json.dumps(state))
    if state["pid"] == os.getpid():
        raise SystemExit("Installation complete. Select Runtime > Restart session, then rerun this cell before continuing.")

    work = Path("/content/rfd4-tutorial-0.0.3")
    work.mkdir(parents=True, exist_ok=True)
    (work / ".project-root").touch()
    RAW = "https://raw.githubusercontent.com/RosettaCommons/RFD4-Proteina/v0.0.3/"
    for rel in ['data/benchmarks/mcsa/mcsa_37/M0664_2dhn.cif.zst']:
        dest = work / rel
        if not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True)
            temporary = dest.with_suffix(dest.suffix + ".download")
            urllib.request.urlretrieve(RAW + rel, temporary)
            temporary.replace(dest)
    os.chdir(work)
    os.environ.setdefault("CLUSTER", "local")
    os.environ.setdefault("DATA_PATH", str(work / "data_path"))

    import torch
    import rfproteina
    import py3Dmol

    if not torch.cuda.is_available() or int((torch.version.cuda or "0").split(".")[0]) != major:
        raise RuntimeError("PyTorch cannot use the selected CUDA profile. Delete the runtime and rerun setup.")
    torch.ones(1, device="cuda").add_(1).item()
    print(f"Ready: PyTorch {torch.__version__}, {torch.cuda.get_device_name(0)}; inputs in {work}")
else:
    print("Not on Colab: using the existing environment.")


In [ ]:
RFD4_GENERATE = Path(sys.executable).with_name("rfd4-generate")
assert RFD4_GENERATE.is_file(), "Run the setup cell first."
print("Generator:", RFD4_GENERATE)


In [ ]:
"""Generate one design from the MCSA enzyme example."""
import os
import shlex
import tempfile
from pathlib import Path

NUM_DESIGNS = 1
SEED = 42

WORK = Path("/content/rfd4-tutorial-0.0.3")
INPUT = WORK / "data/benchmarks/mcsa/mcsa_37/M0664_2dhn.cif.zst"
assert INPUT.is_file(), "Run the setup cell first."

RUN = Path(tempfile.mkdtemp(prefix="colab-enzyme-", dir=WORK))
env = {**os.environ, "CLUSTER": "local", "DATA_PATH": str(WORK / "data_path")}
command = [
    str(RFD4_GENERATE),
    f"inputs={INPUT}",
    f"out_dir={RUN}",
    f"num_replicates={NUM_DESIGNS}",
    f"seed={SEED}",
    "batch_size=1",
    "num_workers=0",
    "compile_lengths=null",      # skip the torch.compile warm-up for a single design
    "compress_outputs=false",
    "~metrics",
    "+metrics.inline_metrics={}",  # no metric is instantiated, so none of their inputs are needed
]
print(shlex.join(command), "\nLog:", RUN / "run.log", flush=True)

with (RUN / "run.log").open("w") as log:
    result = subprocess.run(command, env=env, stdout=log, stderr=subprocess.STDOUT)
if result.returncode:
    print("\n".join((RUN / "run.log").read_text().splitlines()[-40:]))
    result.check_returncode()

designs = sorted((RUN / "inference_samples").rglob("*_generated.cif"))
assert len(designs) == NUM_DESIGNS, f"Expected {NUM_DESIGNS} designs, found {len(designs)}; inspect the log"
for path in designs:
    print(path)


In [ ]:
"""Download the run directory — designs, the effective config and the command log."""
import shutil

from google.colab import files

archive = shutil.make_archive(str(RUN), "zip", root_dir=RUN)
files.download(archive)
